# 22 — NYU compacto: preparação automática no Colab

Nova linha de desenvolvimento, paralela ao notebook 21. Base **NYU Depth
V2 anotada**, com **1.449 imagens internas**. As máscaras humanas de classe
e instância são convertidas automaticamente em caixas 2D da parte visível.

O modelo desta linha cobre **seis classes: cama, sofá, cadeira, mesa,
porta e janela**. A redução de classes faz parte do escopo deste experimento;
não equivale a melhorar o detector anterior de 12 classes.

- Usa o TRAIN oficial de **795 imagens** para separar TRAIN e VAL por cenas.
- Reserva o TEST oficial de **654 imagens**, sem usar previsões dele na escolha.
- Mantém imagens negativas, instâncias pequenas e todas as seis classes.
- Preserva máscaras como fonte, pixels RGB e o registro da conversão.
- Não pede rótulos manuais, formulários, tokens de Kaggle ou caminhos editados.

**Execute este notebook primeiro.** No Colab, CPU é suficiente para esta
preparação. Use **Ambiente de execução → Executar tudo** e autorize a
montagem do seu Google Drive quando o Colab solicitar.

Planeje **15–60 minutos ou mais**, principalmente pelo download oficial
de aproximadamente 2,8 GiB e pela cópia no Drive. Não baixa os 428 GB RAW.
Depois execute o **notebook 23 com GPU**, mantendo o 21 no Kaggle.


In [ ]:
%pip install -q h5py scipy pillow pyyaml


## 1. Carregar o código e montar o Drive


In [ ]:
import os
os.environ['YOLO_OFFLINE']='true'
os.environ['YOLO_CONFIG_DIR']='/tmp/nyu_yolo_config'
os.environ['WANDB_DISABLED']='true'
import sys,json,hashlib,shutil,time
from pathlib import Path
from IPython.display import display,FileLink,Image

BUNDLE='e27b9f628392ab9a3f4c8b4d9d408fcfb368581afcd924b90bf7d9ce53710fc8'
MODULOS={'nyu22_common.py': '"""Experimento compacto NYU: caminhos próprios, fontes e persistência verificável."""\nfrom __future__ import annotations\nfrom pathlib import Path,PurePosixPath\nimport csv,hashlib,json,math,os,shutil,time,urllib.request,zipfile\nimport numpy as np\n\nNAMES=[\'bed\',\'sofa\',\'chair\',\'table\',\'door\',\'window\']\nPT_NAMES=[\'Cama\',\'Sofá\',\'Cadeira\',\'Mesa\',\'Porta\',\'Janela\']\nGLOBAL_IDS=[0,1,2,3,9,8]\nBRANCH=\'TCC_Visao_Computacional/ramo_nyu22_23_v1\'\nSOURCES={\n \'mat\':{\'url\':\'https://horatio.cs.nyu.edu/mit/silberman/nyu_depth_v2/nyu_depth_v2_labeled.mat\',\n        \'sha256\':\'2d724b0c0ab358aa1ce5df855e5bd14a2279ab7202efe22f32a30d612dcb86aa\',\'bytes\':2972037809},\n \'split\':{\'url\':\'https://horatio.cs.nyu.edu/mit/silberman/indoor_seg_sup/splits.mat\',\n          \'sha256\':\'6e081404491a8bfba2f066beaa9713ef6046f9007aa5f799f2a350506a580cee\',\'bytes\':2626},\n \'weight\':{\'url\':\'https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8s.pt\',\n           \'sha256\':\'1f47a78bf100391c2a140b7ac73a1caae18c32779be7d310658112f7ac9aa78a\',\'bytes\':22588772}}\nDATA_SPEC={\'version\':\'22-nyuv2-six-visible-instances-v1\',\'images\':1449,\'official_train\':795,\'official_test\':654,\n    \'width\':640,\'height\':480,\'seed\':22002026,\'val_fraction\':.2,\'split_trials\':128,\n    \'min_train_images\':300,\'min_val_images\':80,\'min_train_gt_class\':10,\'min_val_gt_class\':5,\n    \'phash_distance\':4,\'names\':NAMES,\'bounding_boxes\':\'extensão visível de cada par (rótulo original, instância), sem filtro de tamanho\',\n    \'test_access_during_preparation\':\'conversão de formato e exclusão de duplicatas exatas; nenhuma previsão ou seleção usa TEST\'}\nTRAIN_SPEC={\'version\':\'23-nyuv2-yolov8s-v1\',\'epochs\':80,\'patience\':15,\'imgsz\':640,\'batch\':8,\'workers\':2,\n    \'pretrained_name_aliases\':{\'couch\':\'sofa\',\'dining table\':\'table\'},\n    \'optimizer\':\'AdamW\',\'lr0\':.0005,\'lrf\':.01,\'warmup_epochs\':3.,\'weight_decay\':.0005,\'cos_lr\':True,\'nbs\':32,\n    \'amp\':False,\'mosaic\':.5,\'close_mosaic\':10,\'mixup\':0.,\'cutmix\':0.,\'copy_paste\':0.,\n    \'degrees\':0.,\'translate\':.1,\'scale\':.25,\'fliplr\':.5,\'flipud\':0.,\'hsv_h\':.015,\'hsv_s\':.4,\'hsv_v\':.3,\n    \'seed\':23002026,\'session_hours\':4.,\'inference_conf\':.01,\'nms_iou\':.7,\'match_iou\':.5,\'max_det\':300,\'predict_batch\':8,\n    \'thresholds\':[.05,.10,.15,.20,.25,.30,.40,.50,.60,.70,.80,.90],\n    \'target_precision\':.8,\'target_recall\':.7,\'bootstrap_groups\':1000}\n\ndef clean(v):\n    if isinstance(v,dict):return {str(k):clean(x) for k,x in v.items()}\n    if isinstance(v,(tuple,list)):return [clean(x) for x in v]\n    if isinstance(v,Path):return str(v)\n    if isinstance(v,np.generic):return clean(v.item())\n    if isinstance(v,float) and not math.isfinite(v):return None\n    return v\ndef digest(value):return hashlib.sha256(json.dumps(clean(value),sort_keys=True,ensure_ascii=False,separators=(\',\',\':\'),allow_nan=False).encode()).hexdigest()\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda:f.read(4*1024**2),b\'\'):h.update(block)\n    return h.hexdigest()\ndef save(path,value):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+\'.tmp\')\n    tmp.write_text(json.dumps(clean(value),ensure_ascii=False,indent=2,allow_nan=False),encoding=\'utf-8\');tmp.replace(path)\ndef read(path):return json.loads(Path(path).read_text(encoding=\'utf-8\'))\ndef csv_save(path,rows):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);rows=list(rows)\n    with path.open(\'w\',newline=\'\',encoding=\'utf-8-sig\') as f:\n        fields=list(dict.fromkeys(k for r in rows for k in r));w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)\ndef safe(name):\n    p=PurePosixPath(name)\n    if not name or \'\\\\\' in name or p.is_absolute() or \'..\' in p.parts or \':\' in name:raise ValueError(\'Caminho inválido no pacote: \'+name)\n    return p\ndef extract(archive,root):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        if len(set(z.namelist()))!=len(z.namelist()):raise ValueError(\'ZIP contém nomes repetidos.\')\n        for name in z.namelist():safe(name)\n        z.extractall(root)\ndef zip_paths(target,root,paths):\n    target=Path(target);root=Path(root);tmp=target.with_suffix(\'.tmp\');target.parent.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=3) as z:\n        for path in sorted(set(map(Path,paths))):\n            name=path.relative_to(root).as_posix();safe(name)\n            info=zipfile.ZipInfo(name,(1980,1,1,0,0,0));info.compress_type=zipfile.ZIP_DEFLATED;info.external_attr=0o100644<<16\n            with path.open(\'rb\') as src,z.open(info,\'w\',force_zip64=True) as dst:shutil.copyfileobj(src,dst,2**20)\n    tmp.replace(target);return target\n\ndef download(source,target):\n    target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n    if target.is_file() and target.stat().st_size==source[\'bytes\'] and sha(target)==source[\'sha256\']:return target\n    part=target.with_name(target.name+\'.part\');last=None\n    for attempt in range(3):\n        try:\n            offset=part.stat().st_size if part.exists() else 0\n            if offset>=source[\'bytes\']:part.unlink();offset=0\n            request=urllib.request.Request(source[\'url\'],headers={\'User-Agent\':\'TCC-NYU-22/1.0\',**({\'Range\':f\'bytes={offset}-\'} if offset else {})})\n            with urllib.request.urlopen(request,timeout=60) as response:\n                append=offset>0 and response.status==206\n                if append and not response.headers.get(\'Content-Range\',\'\').startswith(f\'bytes {offset}-\'):raise ValueError(\'Resposta HTTP de retomada incompatível.\')\n                if not append:offset=0\n                done=offset;last_notice=0.\n                with part.open(\'ab\' if append else \'wb\') as f:\n                    while True:\n                        block=response.read(4*1024**2)\n                        if not block:break\n                        f.write(block);done+=len(block)\n                        if time.monotonic()-last_notice>30:\n                            print(f\'Download: {done/1024**2:.0f}/{source["bytes"]/1024**2:.0f} MiB\',flush=True);last_notice=time.monotonic()\n            if part.stat().st_size!=source[\'bytes\']:raise IOError(\'Download incompleto.\')\n            if sha(part)!=source[\'sha256\']:\n                part.replace(part.with_name(part.name+\'.rejeitado\'));raise ValueError(\'SHA-256 diferente da fonte esperada.\')\n            part.replace(target);return target\n        except Exception as exc:\n            last=exc\n            if attempt<2:time.sleep(2**attempt)\n    raise RuntimeError(\'Não foi possível baixar e conferir \'+source[\'url\']+\': \'+str(last)) from last\n\nclass Storage:\n    """Somente a pasta do ramo compacto; nunca sincroniza a raiz do TCC."""\n    def __init__(self,root):self.root=Path(root);self.root.mkdir(parents=True,exist_ok=True)\n    def path(self,relative):safe(relative);return self.root/relative\n    def pull(self,relative,target,expected=None):\n        source=self.path(relative);target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)\n        receipt=source.with_name(source.name+\'.sha256.json\')\n        if expected is None and receipt.is_file():expected=read(receipt)[\'sha256\']\n        if target.is_file() and expected and sha(target)==expected:return target\n        if not source.is_file():raise FileNotFoundError(\'Arquivo ainda não disponível no ramo compacto: \'+relative)\n        tmp=target.with_name(target.name+\'.part\');shutil.copyfile(source,tmp)\n        if expected and sha(tmp)!=expected:tmp.unlink(missing_ok=True);raise ValueError(\'Arquivo remoto incompleto ou alterado: \'+relative)\n        tmp.replace(target);return target\n    def put(self,source,relative):\n        source=Path(source);target=self.path(relative);target.parent.mkdir(parents=True,exist_ok=True)\n        value=sha(source);receipt=target.with_name(target.name+\'.sha256.json\')\n        if target.is_file() and receipt.is_file() and target.stat().st_size==source.stat().st_size:\n            try:\n                if read(receipt)[\'sha256\']==value:return True\n            except (ValueError,KeyError):pass\n        tmp=target.with_name(target.name+\'.tmp\');shutil.copyfile(source,tmp)\n        if sha(tmp)!=value:raise IOError(\'Falha ao conferir a cópia no Drive: \'+relative)\n        tmp.replace(target);save(receipt,{\'sha256\':value,\'bytes\':source.stat().st_size});return True\n    def source(self,key,local,sources=SOURCES):\n        spec=sources[key];name={\'mat\':\'nyu_depth_v2_labeled.mat\',\'split\':\'splits.mat\',\'weight\':\'yolov8s.pt\'}[key]\n        local=Path(local)/name;remote=\'fontes/\'+name\n        if self.path(remote).is_file():return self.pull(remote,local,spec[\'sha256\'])\n        download(spec,local);self.put(local,remote);return local\n\ndef package(output,name):\n    output=Path(output);paths=[];target=output/name\n    for p in sorted(output.rglob(\'*\')):\n        if not p.is_file() or p==target or p.suffix in {\'.tmp\',\'.part\',\'.zip\'}:continue\n        rel=p.relative_to(output)\n        if \'__pycache__\' in rel.parts or \'retomada\' in rel.parts or rel.as_posix().startswith(\'treino/weights/\'):continue\n        if p.suffix==\'.pt\' and rel.as_posix()!=\'melhor_modelo.pt\':continue\n        if p.name==\'integridade.json\':continue\n        paths.append(p)\n    save(output/\'integridade.json\',{\'sha256\':{p.relative_to(output).as_posix():sha(p) for p in paths}})\n    return zip_paths(target,output,paths+[output/\'integridade.json\'])\n', 'nyu22_data.py': '"""Máscaras humanas NYUv2 -> caixas visíveis; separação por cenas e pHash."""\nfrom __future__ import annotations\nfrom collections import Counter,defaultdict\nfrom pathlib import Path\nimport hashlib,re\nimport h5py,numpy as np,yaml\nfrom PIL import Image,ImageDraw\nfrom scipy.fft import dctn\nfrom scipy.io import loadmat\nimport nyu22_common as c\n\ndef strings(f,key):return [\'\'.join(chr(int(v)) for v in f[ref][:].reshape(-1)) for ref in f[key][:].reshape(-1)]\ndef scene_group(name):return re.sub(r\'(?<=\\d)[a-z]+$\',\'\',name.strip().lower())\ndef phash(rgb):\n    gray=np.asarray(Image.fromarray(rgb).convert(\'L\').resize((32,32),Image.Resampling.LANCZOS),dtype=float)\n    low=dctn(gray,norm=\'ortho\')[:8,:8];bits=low>np.median(low.ravel()[1:]);bits[0,0]=False\n    return int.from_bytes(np.packbits(bits.ravel()).tobytes(),\'big\')\n\ndef boxes_from_masks(labels,instances,name_to_id):\n    if labels.shape!=instances.shape or labels.ndim!=2:raise ValueError(\'Máscaras incompatíveis.\')\n    result=[]\n    for cid,name in enumerate(c.NAMES):\n        raw_id=name_to_id[name];selected=labels==raw_id\n        for instance in np.unique(instances[selected]):\n            # Mesmo critério do get_instance_masks.m oficial: rótulo + instância.\n            yy,xx=np.where(selected&(instances==instance))\n            if not len(xx):continue\n            result.append({\'class_id\':cid,\'class\':name,\'bbox\':[int(xx.min()),int(yy.min()),int(xx.max()+1),int(yy.max()+1)],\n                \'mask_pixels\':int(len(xx)),\'raw_label\':int(raw_id),\'instance\':int(instance)})\n    return result\ndef yolo_bytes(boxes,w,h):\n    lines=[]\n    for obj in boxes:\n        x1,y1,x2,y2=obj[\'bbox\']\n        if not(0<=x1<x2<=w and 0<=y1<y2<=h):raise ValueError(\'Caixa derivada inválida.\')\n        lines.append(f\'{obj["class_id"]} {(x1+x2)/(2*w):.17g} {(y1+y2)/(2*h):.17g} {(x2-x1)/w:.17g} {(y2-y1)/h:.17g}\\n\')\n    return \'\'.join(lines).encode()\n\ndef convert(f,index,role,root,name_to_id,scenes,spec):\n    i=index-1;rgb=np.asarray(f[\'images\'][i]).transpose(2,1,0)\n    labels=np.asarray(f[\'labels\'][i]).T;instances=np.asarray(f[\'instances\'][i]).T\n    if rgb.shape!=(spec[\'height\'],spec[\'width\'],3) or labels.shape!=rgb.shape[:2]:raise ValueError(\'Orientação ou dimensão NYU inesperada.\')\n    boxes=boxes_from_masks(labels,instances,name_to_id);iid=f\'nyu_{index:05d}\'\n    image=Path(root)/\'images\'/role/(iid+\'.png\');label=Path(root)/\'labels\'/role/(iid+\'.txt\')\n    image.parent.mkdir(parents=True,exist_ok=True);label.parent.mkdir(parents=True,exist_ok=True)\n    Image.fromarray(rgb).save(image);label.write_bytes(yolo_bytes(boxes,spec[\'width\'],spec[\'height\']))\n    return {\'id\':iid,\'index\':index,\'role\':role,\'scene\':scenes[i],\'scene_group\':scene_group(scenes[i]),\n        \'image\':image.relative_to(root).as_posix(),\'label\':label.relative_to(root).as_posix(),\n        \'image_sha256\':c.sha(image),\'label_sha256\':c.sha(label),\'pixel_sha256\':hashlib.sha256(rgb.tobytes()).hexdigest(),\n        \'width\':spec[\'width\'],\'height\':spec[\'height\'],\'phash\':phash(rgb),\'gt\':boxes,\n        \'unlabeled_fraction\':float((labels==0).mean())}\n\ndef groups_for(rows,spec):\n    parent=list(range(len(rows)))\n    def find(i):\n        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]\n        return i\n    def union(i,j):parent[find(j)]=find(i)\n    scenes={};bands=[defaultdict(list) for _ in range(8)];similar=[]\n    for i,r in enumerate(rows):\n        if r[\'scene_group\'] in scenes:union(i,scenes[r[\'scene_group\']])\n        scenes.setdefault(r[\'scene_group\'],i)\n        candidates={j for k in range(8) for j in bands[k][(r[\'phash\']>>(8*k))&255]}\n        for j in candidates:\n            if (r[\'phash\']^rows[j][\'phash\']).bit_count()<=spec[\'phash_distance\']:\n                union(i,j)\n                if r[\'scene_group\']!=rows[j][\'scene_group\']:similar.append([r[\'id\'],rows[j][\'id\']])\n        for k in range(8):bands[k][(r[\'phash\']>>(8*k))&255].append(i)\n    groups=defaultdict(list)\n    for i,r in enumerate(rows):groups[find(i)].append(r)\n    for group in groups.values():\n        key=min(r[\'id\'] for r in group)\n        for r in group:r[\'split_group\']=key\n    return sorted(groups.values(),key=lambda g:min(r[\'id\'] for r in g)),similar\n\ndef split_development(rows,spec):\n    groups,similar=groups_for(rows,spec);total=np.bincount([o[\'class_id\'] for r in rows for o in r[\'gt\']],minlength=6)\n    options=[];n=max(1,round(len(groups)*spec[\'val_fraction\']))\n    for trial in range(spec[\'split_trials\']):\n        order=np.random.default_rng(spec[\'seed\']+trial).permutation(len(groups));v={int(x) for x in order[:n]}\n        val=[r for i,g in enumerate(groups) if i in v for r in g];train=[r for i,g in enumerate(groups) if i not in v for r in g]\n        counts=np.bincount([o[\'class_id\'] for r in val for o in r[\'gt\']],minlength=6)\n        if len(train)<spec[\'min_train_images\'] or len(val)<spec[\'min_val_images\']:continue\n        if (counts<spec[\'min_val_gt_class\']).any() or ((total-counts)<spec[\'min_train_gt_class\']).any():continue\n        score=abs(len(val)/len(rows)-spec[\'val_fraction\'])+float(np.mean(abs(counts/np.maximum(total,1)-spec[\'val_fraction\'])))\n        options.append((score,trial,train,val))\n    if not options:raise ValueError(\'Não foi possível obter TRAIN/VAL com cenas separadas e suporte mínimo nas seis classes.\')\n    _,trial,train,val=min(options,key=lambda x:(x[0],x[1]))\n    for role,group in [(\'train\',train),(\'val\',val)]:\n        for r in group:r[\'role\']=role\n    return sorted(train,key=lambda r:r[\'id\']),sorted(val,key=lambda r:r[\'id\']),{\'trial\':trial,\'groups\':len(groups),\'phash_pairs_across_scenes\':similar,\n        \'criterion\':\'Proporções de imagens e caixas, antes de qualquer treinamento; somente TRAIN oficial. Não usa precisão, recall ou rótulos de TEST.\'}\n\ndef relocate(rows,root):\n    for r in rows:\n        for key,folder in [(\'image\',\'images\'),(\'label\',\'labels\')]:\n            old=Path(root)/r[key];new=Path(root)/folder/r[\'role\']/old.name\n            if old!=new:new.parent.mkdir(parents=True,exist_ok=True);old.replace(new);r[key]=new.relative_to(root).as_posix()\ndef contact_sheet(rows,root,target):\n    selected=rows[:12];page=Image.new(\'RGB\',(900,math_ceil(len(selected)/3)*250),\'white\');draw=ImageDraw.Draw(page)\n    for i,r in enumerate(selected):\n        im=Image.open(Path(root)/r[\'image\']).convert(\'RGB\');d=ImageDraw.Draw(im)\n        for o in r[\'gt\']:\n            d.rectangle(o[\'bbox\'],outline=\'lime\',width=2);d.text(o[\'bbox\'][:2],c.PT_NAMES[o[\'class_id\']],fill=\'yellow\')\n        im.thumbnail((294,220));x=(i%3)*300;y=(i//3)*250;page.paste(im,(x,y));draw.text((x,y+222),r[\'id\']+\' | \'+r[\'role\'],fill=\'black\')\n    page.save(target)\ndef math_ceil(v):return int(np.ceil(v))\n\ndef prepare(mat,split,root,output,spec=c.DATA_SPEC):\n    root=Path(root);output=Path(output);root.mkdir(parents=True,exist_ok=True);output.mkdir(parents=True,exist_ok=True)\n    split=loadmat(split);train_ids=[int(x) for x in split[\'trainNdxs\'].reshape(-1)];test_ids=[int(x) for x in split[\'testNdxs\'].reshape(-1)]\n    if len(train_ids)!=spec[\'official_train\'] or len(test_ids)!=spec[\'official_test\'] or set(train_ids)&set(test_ids) or set(train_ids+test_ids)!=set(range(1,spec[\'images\']+1)):\n        raise ValueError(\'Divisão oficial incompleta ou sobreposta.\')\n    with h5py.File(mat,\'r\') as f:\n        names=strings(f,\'names\');scenes=strings(f,\'scenes\')\n        mapping={n:names.index(n)+1 for n in c.NAMES}\n        if len(scenes)!=spec[\'images\'] or f[\'images\'].shape!=(spec[\'images\'],3,spec[\'width\'],spec[\'height\']):raise ValueError(\'Esquema NYU diferente da fonte esperada.\')\n        train_scenes={scene_group(scenes[i-1]) for i in train_ids};test_scenes={scene_group(scenes[i-1]) for i in test_ids}\n        if train_scenes&test_scenes:raise ValueError(\'Cenas da reserva aparecem no TRAIN oficial.\')\n        pool=[]\n        for k,index in enumerate(train_ids,1):\n            pool.append(convert(f,index,\'pool\',root,mapping,scenes,spec))\n            if k%100==0:print(f\'Conversão do TRAIN oficial: {k}/{len(train_ids)}\',flush=True)\n        train,val,allocation=split_development(pool,spec);relocate(train+val,root)\n        reserve=[];pixels={r[\'pixel_sha256\'] for r in pool}\n        for k,index in enumerate(test_ids,1):\n            r=convert(f,index,\'test\',root,mapping,scenes,spec)\n            if r[\'pixel_sha256\'] in pixels:raise ValueError(\'Imagem da reserva coincide exatamente com o desenvolvimento.\')\n            reserve.append(r)\n            if k%100==0:print(f\'Conversão da reserva, sem avaliação: {k}/{len(test_ids)}\',flush=True)\n    if {r[\'split_group\'] for r in train}&{r[\'split_group\'] for r in val}:raise ValueError(\'Grupo repetido entre TRAIN e VAL.\')\n    counts=[{\'split\':role,\'class\':name,\'gt\':sum(o[\'class_id\']==cid for r in rows for o in r[\'gt\']),\n             \'images\':sum(any(o[\'class_id\']==cid for o in r[\'gt\']) for r in rows)}\n            for role,rows in [(\'train\',train),(\'val\',val)] for cid,name in enumerate(c.NAMES)]\n    report={\'version\':spec[\'version\'],\'train_images\':len(train),\'val_images\':len(val),\'test_images_reserved\':len(reserve),\n        \'train_scenes\':len({r[\'scene_group\'] for r in train}),\'val_scenes\':len({r[\'scene_group\'] for r in val}),\'test_scenes\':len(test_scenes),\n        \'scene_overlap_train_val\':0,\'scene_overlap_development_test\':0,\'class_counts_development\':counts,\'allocation\':allocation,\n        \'raw_label_ids\':mapping,\'unlabeled_pixel_fraction_development\':float(np.mean([r[\'unlabeled_fraction\'] for r in pool])),\n        \'classes\':c.NAMES,\'global_ids_old_project\':c.GLOBAL_IDS,\'manual_labels_required\':False,\'test_used_for_selection\':False,\n        \'task\':\'Detecção 2D em seis categorias NYU; caixas derivadas das máscaras visíveis, sem profundidade como entrada.\',\n        \'limits\':[\'Escopo e referência distintos do experimento 21; percentuais não são diretamente comparáveis.\',\n                  \'SUN RGB-D incorpora imagens NYU: não reutilizar pesos Home/C10/21 nem declarar comparação independente com eles.\',\n                  \'Pixels sem rótulo existem na origem. As caixas representam os objetos anotados, não uma garantia de anotação perfeita.\']}\n    manifest={\'version\':spec[\'version\'],\'classes\':c.NAMES,\'train\':train,\'val\':val,\'test_ids\':test_ids,\'test_scenes\':sorted(test_scenes),\'report\':report}\n    c.save(root/\'manifesto_desenvolvimento.json\',manifest);c.save(root/\'manifesto_reserva.json\',{\'classes\':c.NAMES,\'test\':reserve})\n    c.save(output/\'preparacao_nyu22.json\',report);c.csv_save(output/\'classes_desenvolvimento.csv\',counts)\n    contact_sheet(train[:6]+val[:6],root,output/\'exemplos_desenvolvimento.jpg\')\n    dev_paths=[root/\'manifesto_desenvolvimento.json\']+[root/r[k] for r in train+val for k in [\'image\',\'label\']]\n    test_paths=[root/\'manifesto_reserva.json\']+[root/r[k] for r in reserve for k in [\'image\',\'label\']]\n    dev=c.zip_paths(output/\'desenvolvimento_nyu22.zip\',root,dev_paths);test=c.zip_paths(output/\'reserva_nyu22.zip\',root,test_paths)\n    return report,dev,test\n\ndef load_development(archive,root):\n    root=Path(root);c.extract(archive,root);m=c.read(root/\'manifesto_desenvolvimento.json\')\n    if m[\'classes\']!=c.NAMES:raise ValueError(\'Classes da base preparada diferentes.\')\n    train=m[\'train\'];val=m[\'val\']\n    if any(r[\'role\']!=\'train\' for r in train) or any(r[\'role\']!=\'val\' for r in val):raise ValueError(\'Papéis trocados na base.\')\n    if {r[\'index\'] for r in train+val}&set(m[\'test_ids\']):raise ValueError(\'TEST entrou no desenvolvimento.\')\n    if {r[\'split_group\'] for r in train}&{r[\'split_group\'] for r in val}:raise ValueError(\'Grupos sobrepostos.\')\n    hydrate(train+val,root)\n    yaml_file=write_yaml(train,val,root,\'data.yaml\')\n    return train,val,yaml_file,m\ndef hydrate(rows,root):\n    for r in rows:\n        for key,hkey in [(\'image\',\'image_sha256\'),(\'label\',\'label_sha256\')]:\n            c.safe(r[key]);p=Path(root)/r[key]\n            if c.sha(p)!=r[hkey]:raise ValueError(\'Imagem ou rótulo preparado foi alterado: \'+r[\'id\'])\n            r[key]=p\ndef write_yaml(train,val,root,name):\n    root=Path(root);a=root/(name+\'.train.txt\');b=root/(name+\'.val.txt\')\n    a.write_text(\'\\n\'.join(str(Path(r[\'image\']).resolve()) for r in train)+\'\\n\');b.write_text(\'\\n\'.join(str(Path(r[\'image\']).resolve()) for r in val)+\'\\n\')\n    target=root/name;target.write_text(yaml.safe_dump({\'path\':str(root.resolve()),\'train\':str(a.resolve()),\'val\':str(b.resolve()),\'names\':dict(enumerate(c.NAMES))},sort_keys=False));return target\ndef load_reserve(archive,root,development_manifest):\n    root=Path(root);c.extract(archive,root);m=c.read(root/\'manifesto_reserva.json\');rows=m[\'test\']\n    if m[\'classes\']!=c.NAMES or any(r[\'role\']!=\'test\' for r in rows):raise ValueError(\'Reserva incompatível.\')\n    if {r[\'index\'] for r in rows}!=set(development_manifest[\'test_ids\']):raise ValueError(\'IDs de TEST foram substituídos.\')\n    protected=development_manifest[\'train\']+development_manifest[\'val\']\n    if {r[\'pixel_sha256\'] for r in rows}&{r[\'pixel_sha256\'] for r in protected}:raise ValueError(\'Pixels repetidos entre desenvolvimento e reserva.\')\n    if {r[\'scene_group\'] for r in rows}&{r[\'scene_group\'] for r in protected}:raise ValueError(\'Cenas da reserva sobrepostas.\')\n    hydrate(rows,root);return rows\n', 'nyu23_eval.py': '"""Seleção em VAL e teste congelado; métricas micro por objeto com IoU 0,5."""\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom collections import defaultdict\nimport gzip,json,math\nimport numpy as np\nimport nyu22_common as c\n\ndef metrics(tp,fn,fp):\n    tp,fn,fp=map(int,(tp,fn,fp));p=tp/(tp+fp) if tp+fp else None;r=tp/(tp+fn) if tp+fn else None\n    return {\'tp\':tp,\'fn\':fn,\'fp\':fp,\'precision\':p,\'recall\':r,\'f1\':2*tp/(2*tp+fn+fp) if 2*tp+fn+fp else None}\ndef match(gt,pred,threshold=.5):\n    if not gt or not pred:return [],list(range(len(gt))),list(range(len(pred)))\n    a=np.asarray([o[\'bbox\'] for o in gt],float);b=np.asarray([o[\'bbox\'] for o in pred],float)\n    lo=np.maximum(a[:,None,:2],b[None,:,:2]);hi=np.minimum(a[:,None,2:],b[None,:,2:])\n    inter=np.prod(np.maximum(hi-lo,0),axis=2);aa=np.prod(np.maximum(a[:,2:]-a[:,:2],0),axis=1);bb=np.prod(np.maximum(b[:,2:]-b[:,:2],0),axis=1)\n    den=aa[:,None]+bb[None,:]-inter;overlap=np.divide(inter,den,out=np.zeros_like(inter),where=den>0)\n    candidates=sorted([(-float(overlap[i,j]),i,j) for i,g in enumerate(gt) for j,p in enumerate(pred) if g[\'class_id\']==p[\'class_id\'] and overlap[i,j]>=threshold])\n    gs=set();ps=set();pairs=[]\n    for neg,i,j in candidates:\n        if i not in gs and j not in ps:gs.add(i);ps.add(j);pairs.append((i,j,-neg))\n    return pairs,[i for i in range(len(gt)) if i not in gs],[j for j in range(len(pred)) if j not in ps]\n\ndef profile(rows,cache,threshold,spec=c.TRAIN_SPEC):\n    totals=np.zeros(3,dtype=np.int64);classes=np.zeros((6,3),dtype=np.int64);details=[]\n    for row in rows:\n        pred=[p for p in cache[row[\'id\']] if p[\'confidence\']>=threshold];gt=row[\'gt\']\n        pairs,missing,extras=match(gt,pred,spec[\'match_iou\']);counts=[len(pairs),len(missing),len(extras)];totals+=counts\n        for gi,_,_ in pairs:classes[gt[gi][\'class_id\'],0]+=1\n        for gi in missing:classes[gt[gi][\'class_id\'],1]+=1\n        for pi in extras:classes[pred[pi][\'class_id\'],2]+=1\n        details.append({\'id\':row[\'id\'],\'scene_group\':row[\'scene_group\'],\'tp\':counts[0],\'fn\':counts[1],\'fp\':counts[2]})\n    det=metrics(*totals)\n    score=min((det[\'precision\'] or 0)/spec[\'target_precision\'],(det[\'recall\'] or 0)/spec[\'target_recall\'])+1e-6*(det[\'f1\'] or 0)\n    return {\'images\':len(rows),\'threshold\':threshold,\'detector\':det,\'fitness\':score,\n        \'target_met\':bool(det[\'precision\'] is not None and det[\'recall\'] is not None and det[\'precision\']>=spec[\'target_precision\'] and det[\'recall\']>=spec[\'target_recall\']),\n        \'classes\':[{\'class_id\':i,\'class\':name,\'gt\':int(classes[i,0]+classes[i,1]),**metrics(*classes[i])} for i,name in enumerate(c.NAMES)],\'image_rows\':details}\ndef choose(profiles):return max(profiles,key=lambda v:(v[\'fitness\'],-v[\'threshold\']))\n\ndef predict(model,rows,device,spec):\n    if list(model.names.values())!=c.NAMES:raise ValueError(\'O peso não contém as seis classes na ordem esperada.\')\n    batch=spec[\'predict_batch\'];cache={};diagnostic={\'zero_area\':0,\'outside\':0,\'saturated_images\':0,\'coordinates_changed\':False,\'predictions_removed\':0}\n    for start in range(0,len(rows),batch):\n        block=rows[start:start+batch]\n        result=model.predict([str(r[\'image\']) for r in block],conf=spec[\'inference_conf\'],imgsz=spec[\'imgsz\'],rect=True,\n            iou=spec[\'nms_iou\'],max_det=spec[\'max_det\'],quantize=32,batch=batch,device=device,verbose=False,save=False)\n        if len(result)!=len(block):raise ValueError(\'Número de resultados diferente do bloco de imagens.\')\n        for r,out in zip(block,result):\n            if Path(out.path).resolve()!=Path(r[\'image\']).resolve() or tuple(out.orig_shape)!=(r[\'height\'],r[\'width\']):raise ValueError(\'Previsão pertence a outra imagem.\')\n            values=[]\n            if out.boxes is not None:\n                for cls,xy,confidence in zip(out.boxes.cls.cpu().numpy(),out.boxes.xyxy.cpu().numpy(),out.boxes.conf.cpu().numpy()):\n                    cid=int(cls);xy=[float(x) for x in xy];confidence=float(confidence)\n                    if not(0<=cid<6 and all(math.isfinite(x) for x in xy+[confidence]) and 0<=confidence<=1 and xy[2]>=xy[0] and xy[3]>=xy[1]):raise ValueError(\'Previsão estruturalmente inválida.\')\n                    diagnostic[\'zero_area\']+=int(xy[2]==xy[0] or xy[3]==xy[1]);diagnostic[\'outside\']+=int(min(xy)<-.01 or xy[2]>r[\'width\']+.01 or xy[3]>r[\'height\']+.01)\n                    values.append({\'class_id\':cid,\'bbox\':xy,\'confidence\':confidence})\n            cache[r[\'id\']]=values;diagnostic[\'saturated_images\']+=int(len(values)==spec[\'max_det\'])\n    return cache,diagnostic\n\ndef obtain_cache(model,rows,device,spec,output,weight_sha,role,checkpoint=None):\n    if any(r[\'role\']!=role for r in rows):raise ValueError(\'Papel proibido nesta avaliação.\')\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    request={\'role\':role,\'weight_sha256\':weight_sha,\'input_sha256\':c.digest([{k:r[k] for k in [\'id\',\'image_sha256\',\'label_sha256\',\'width\',\'height\']} for r in rows]),\n        \'inference\':{k:spec[k] for k in [\'imgsz\',\'inference_conf\',\'nms_iou\',\'max_det\',\'predict_batch\']},\'quantize\':32}\n    key=c.digest(request);output=output/key[:24];output.mkdir(parents=True,exist_ok=True);cache={};diagnostics=[]\n    for start in range(0,len(rows),32):\n        block=rows[start:start+32];path=output/f\'bloco_{start//32:04d}.json.gz\'\n        if path.is_file():\n            with gzip.open(path,\'rt\') as f:payload=json.load(f)\n            if payload[\'request_key\']!=key or payload[\'data_sha256\']!=c.digest(payload[\'data\']) or list(payload[\'data\'])!=[r[\'id\'] for r in block]:raise ValueError(\'Cache de previsão alterado ou de outra execução.\')\n        else:\n            data,diagnostic=predict(model,block,device,spec);payload={\'request_key\':key,\'data\':data,\'data_sha256\':c.digest(data),\'diagnostic\':diagnostic}\n            tmp=path.with_suffix(\'.tmp\')\n            with gzip.open(tmp,\'wt\',encoding=\'utf-8\') as f:json.dump(payload,f,ensure_ascii=False,allow_nan=False)\n            tmp.replace(path)\n        cache.update(payload[\'data\']);diagnostics.append(payload[\'diagnostic\'])\n        print(f\'Avaliação {role}: {start+len(block)}/{len(rows)} imagens\',flush=True)\n        if checkpoint:checkpoint()\n    c.save(output/\'manifesto_cache.json\',{\'request\':request,\'request_key\':key,\'complete\':True,\'diagnostics\':diagnostics})\n    return cache\n\ndef evaluate_development(model,rows,device,spec=c.TRAIN_SPEC,output=None,weight_sha=None,fixed_threshold=None):\n    if any(r[\'role\']!=\'val\' for r in rows):raise ValueError(\'A seleção admite somente VAL.\')\n    if output is None:cache,_=predict(model,rows,device,spec)\n    else:cache=obtain_cache(model,rows,device,spec,Path(output)/\'cache\',weight_sha,\'val\')\n    profiles=[profile(rows,cache,t,spec) for t in spec[\'thresholds\']]\n    selected=choose(profiles) if fixed_threshold is None else profile(rows,cache,fixed_threshold,spec)\n    if output is not None:\n        c.save(Path(output)/\'metricas.json\',selected);c.save(Path(output)/\'grade_limiares.json\',profiles)\n        c.csv_save(Path(output)/\'classes.csv\',selected[\'classes\']);c.csv_save(Path(output)/\'imagens.csv\',selected[\'image_rows\'])\n    return selected,profiles\n\ndef scene_bootstrap(image_rows,spec=c.TRAIN_SPEC):\n    groups=defaultdict(lambda:np.zeros(3,dtype=np.int64))\n    for r in image_rows:groups[r[\'scene_group\']]+=np.asarray([r[\'tp\'],r[\'fn\'],r[\'fp\']])\n    values=np.array(list(groups.values()));rng=np.random.default_rng(spec[\'seed\']);samples={k:[] for k in [\'precision\',\'recall\',\'f1\']}\n    for _ in range(spec[\'bootstrap_groups\']):\n        result=metrics(*values[rng.integers(0,len(values),size=len(values))].sum(axis=0))\n        for k in samples:\n            if result[k] is not None:samples[k].append(result[k])\n    return {\'method\':\'Bootstrap de cenas, não de caixas individuais\',\'groups\':len(values),\'replicates\':spec[\'bootstrap_groups\'],\n        \'interval_95\':{k:[float(x) for x in np.quantile(v,[.025,.975])] if v else None for k,v in samples.items()}}\n', 'nyu23_report.py': '"""Relato separado: seis classes, máscaras visíveis e reserva própria desta linha."""\nfrom pathlib import Path\nimport html\nimport nyu22_common as c\n\ndef pct(v):return \'indefinido\' if v is None else f\'{100*v:.2f}%\'.replace(\'.\',\',\')\ndef table(rows):\n    if not rows:return \'<p>Ainda não avaliado.</p>\'\n    keys=list(rows[0]);return \'<div class="scroll"><table><tr>\'+\'\'.join(\'<th>\'+html.escape(k)+\'</th>\' for k in keys)+\'</tr>\'+\'\'.join(\'<tr>\'+\'\'.join(\'<td>\'+html.escape(str(r.get(k,\'\')))+\'</td>\' for k in keys)+\'</tr>\' for r in rows)+\'</table></div>\'\ndef build(job,run,best,val,test,spec=c.TRAIN_SPEC):\n    output=Path(job[\'output\']);prep=job[\'info\'][\'report\']\n    status=\'pausado_retomavel\' if not run[\'completed\'] else \'concluido_meta_atingida_teste\' if test and test[\'target_met\'] else \'concluido_meta_nao_atingida_teste\'\n    summary={\'version\':spec[\'version\'],\'completed\':bool(run[\'completed\'] and test is not None),\'status\':status,\'run\':run,\n        \'context\':job[\'context\'],\'dataset_id\':job[\'info\'][\'dataset_id\'],\'classes\':c.NAMES,\'class_names_pt\':c.PT_NAMES,\'old_project_class_ids\':c.GLOBAL_IDS,\n        \'model\':\'yolov8s\',\'best_epoch\':best[\'epoch\'],\'weight_sha256\':best[\'weight_sha256\'],\'threshold\':val[\'threshold\'],\n        \'goal\':{\'precision\':spec[\'target_precision\'],\'recall\':spec[\'target_recall\']},\'validation\':val,\'test\':test,\n        \'target_met_validation\':val[\'target_met\'],\'target_met_test\':test[\'target_met\'] if test else None,\n        \'test_used_for_selection\':False,\'same_benchmark_as_21\':False,\'existing_model_replaced\':False,\'manual_labels_required\':False,\n        \'train_images\':prep[\'train_images\'],\'val_images\':prep[\'val_images\'],\'test_images\':prep[\'test_images_reserved\'],\n        \'limits\':prep[\'limits\']+[\'Atingir a meta nesta base e em seis classes não garante o mesmo resultado nas 12 classes nem em fotos do usuário.\',\n            \'Treino usa somente RGB; as caixas vêm de máscaras humanas de partes visíveis. Não são caixas amodais de objetos completos.\',\n            \'Não valida distância nem segurança de locomoção.\']}\n    c.save(output/\'resumo_nyu23.json\',summary)\n    comparison=[];classes=[]\n    for label,s in [(\'Validação\',val),(\'Teste reservado\',test)]:\n        if s is None:continue\n        comparison.append({\'Conjunto\':label,\'Imagens\':s[\'images\'],\'Precisão\':pct(s[\'detector\'][\'precision\']),\'Recall\':pct(s[\'detector\'][\'recall\']),\'F1\':pct(s[\'detector\'][\'f1\']),\'Meta 80/70\':\'sim\' if s[\'target_met\'] else \'não\'})\n        for row in s[\'classes\']:classes.append({\'Conjunto\':label,\'Classe\':c.PT_NAMES[row[\'class_id\']],\'GT\':row[\'gt\'],\'Precisão\':pct(row[\'precision\']),\'Recall\':pct(row[\'recall\']),\'F1\':pct(row[\'f1\']),\'TP / FN / FP\':f\'{row["tp"]} / {row["fn"]} / {row["fp"]}\'})\n    c.csv_save(output/\'comparacao_val_teste.csv\',comparison);c.csv_save(output/\'classes_val_teste.csv\',classes)\n    message=\'Treino pausado; execute Run All novamente para retomar. A reserva não foi avaliada.\' if not run[\'completed\'] else \'Treino e teste concluídos. Meta conjunta atingida no teste desta linha.\' if test and test[\'target_met\'] else \'Treino e teste concluídos. Meta conjunta não atingida no teste desta linha.\'\n    text=f\'\'\'# Rodada compacta NYU — seis classes\n\n{message}\n\nO detector YOLOv8s foi treinado a partir de pesos públicos COCO, em RGB, para cama, sofá, cadeira, mesa, porta e janela. A base contém 1.449 imagens; nesta preparação ficaram {prep[\'train_images\']} em TRAIN, {prep[\'val_images\']} em VAL e {prep[\'test_images_reserved\']} no TEST oficial. TRAIN/VAL foram separados em grupos de cenas e semelhança perceptual dentro do TRAIN oficial. A escolha da divisão usou representatividade de rótulos, antes de treinar, sem resultados de modelos ou rótulos de TEST.\n\nAs caixas foram obtidas automaticamente da extensão visível de cada par de classe original e instância nas máscaras humanas NYU, incluindo instância zero quando existe, como no toolbox original. Não houve anotação manual nem criação de rótulos por previsão. Imagens sem objetos das seis classes foram mantidas. Pixels sem rótulo existem na origem.\n\nForam concluídas {run[\'epochs\']} épocas; a melhor selecionada foi {best[\'epoch\']}. A confiança escolhida na validação foi {val[\'threshold\']}. O critério tenta atender precisão 80% e recall 70% simultaneamente, com IoU 0,5 e as seis classes mantidas. No teste, peso e confiança permaneceram fixos. O teste só acontece após o fim do treino; uma execução pausada não abre suas previsões.\n\n\'\'\'\n    text+=\'| Conjunto | Precisão | Recall | F1 |\\n| --- | ---: | ---: | ---: |\\n\'\n    for r in comparison:text+=f\'| {r["Conjunto"]} | {r["Precisão"]} | {r["Recall"]} | {r["F1"]} |\\n\'\n    text+=\'\\nOs percentuais desta rodada não são comparáveis diretamente aos da rodada 21: mudaram base, classes e definição das caixas. SUN RGB-D incorpora imagens NYU; esta linha começa de pesos públicos e não usa pesos Home/C10/21. A reserva serve para a avaliação desta nova linha, não para afirmar independência de versões anteriores treinadas em Home/SUN.\\n\'\n    if test:text+=f\'\\nAP padrão Ultralytics: mAP50={pct(test["native_ap"]["map50"])}; mAP50–95={pct(test["native_ap"]["map50_95"])}. AP usa uma curva de confiança e não equivale à precisão no limiar operacional. Intervalos de 95% por reamostragem de cenas constam no JSON; a indicação de meta usa a estimativa pontual.\\n\'\n    (output/\'texto_resultados_nyu23.md\').write_text(text,encoding=\'utf-8\')\n    page=\'<!doctype html><html lang="pt-BR"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>NYU compacto</title><style>body{font:16px/1.6 system-ui;margin:0;background:#eef2f6;color:#173046}main{max-width:1300px;margin:auto;padding:28px}.note{background:white;padding:20px;border:1px solid #cbd5e1;border-radius:12px}.scroll{overflow:auto}table{border-collapse:collapse;background:white;width:100%}th,td{padding:9px;border:1px solid #cbd5e1;text-align:left}</style><main><h1>NYU: detector de seis classes</h1>\'\n    page+=\'<div class="note"><strong>\'+html.escape(message)+\'</strong><p>\'+f\'Épocas: {run["epochs"]}. Melhor época: {best["epoch"]}. Confiança: {val["threshold"]}.\'+\'</p></div>\'\n    page+=\'<h2>Validação e teste</h2>\'+table(comparison)+\'<h2>Resultados por classe</h2>\'+table(classes)\n    page+=\'<h2>Limitações</h2><ul>\'+\'\'.join(\'<li>\'+html.escape(t)+\'</li>\' for t in summary[\'limits\'])+\'</ul><p>Resultados em pacote_treino_nyu23.zip. Não houve substituição automática do modelo anterior.</p></main></html>\'\n    (output/\'relatorio_nyu23.html\').write_text(page,encoding=\'utf-8\');return summary\n', 'nyu23_training.py': '"""YOLOv8s padrão, seleção VAL e retomada completa em dois slots."""\nfrom __future__ import annotations\nfrom copy import deepcopy\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom contextlib import contextmanager\nimport gc,json,math,random,shutil,time\nimport numpy as np\nimport torch\nimport ultralytics\nfrom ultralytics import YOLO\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel\nimport nyu22_common as c\nimport nyu23_eval as ev\n\n@contextmanager\ndef isolated():\n    py=random.getstate();np_state=np.random.get_state();cpu=torch.get_rng_state();cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n    threads=torch.get_num_threads();det=torch.are_deterministic_algorithms_enabled();warn=torch.is_deterministic_algorithms_warn_only_enabled();flags=(torch.backends.cudnn.benchmark,torch.backends.cudnn.deterministic)\n    try:yield\n    finally:\n        random.setstate(py);np.random.set_state(np_state);torch.set_rng_state(cpu)\n        if cuda is not None:torch.cuda.set_rng_state_all(cuda)\n        torch.set_num_threads(threads);torch.use_deterministic_algorithms(det,warn_only=warn);torch.backends.cudnn.benchmark,torch.backends.cudnn.deterministic=flags\n\ndef export(source,target):\n    obj=torch.load(source,map_location=\'cpu\',weights_only=False);model=obj.get(\'ema\') or obj.get(\'model\')\n    if type(model) is not DetectionModel or list(model.names.values())!=c.NAMES:raise ValueError(\'Checkpoint não é DetectionModel padrão com seis classes.\')\n    model.criterion=None;model=model.float().eval()\n    if any(not torch.isfinite(v).all() for v in model.state_dict().values() if v.is_floating_point()):raise FloatingPointError(\'Peso de inferência não finito.\')\n    obj.update(model=model,ema=None,optimizer=None);target=Path(target);target.parent.mkdir(parents=True,exist_ok=True);torch.save(obj,target)\n\nclass Validator:\n    def __init__(self,trainer):\n        self.args=deepcopy(trainer.args);self.metrics=SimpleNamespace(keys=[\'dev/precision\',\'dev/recall\',\'dev/f1\',\'dev/threshold\',\'dev/target_met\'])\n    def __call__(self,trainer):\n        with isolated():\n            wrapper=YOLO(\'yolov8s.yaml\');wrapper.model=deepcopy(trainer.ema.ema).float().eval();wrapper.predictor=None;wrapper.task=\'detect\'\n            try:selected,_=ev.evaluate_development(wrapper,trainer.rows,trainer.device,trainer.spec)\n            finally:del wrapper;gc.collect()\n        trainer.pending=selected\n        return {\'dev/precision\':selected[\'detector\'][\'precision\'] or 0.,\'dev/recall\':selected[\'detector\'][\'recall\'] or 0.,\n            \'dev/f1\':selected[\'detector\'][\'f1\'] or 0.,\'dev/threshold\':selected[\'threshold\'],\'dev/target_met\':int(selected[\'target_met\']),\'fitness\':selected[\'fitness\']}\n\nclass Trainer(DetectionTrainer):\n    spec=c.TRAIN_SPEC;rows=None;output=None;context=None;backup=None\n    def __init__(self,*args,**kwargs):\n        self.session_started=time.monotonic();self.epoch_started=None;self.pending=None;self.reason=None\n        if ultralytics.__version__!=\'8.4.150\':raise RuntimeError(\'Use ultralytics==8.4.150.\')\n        super().__init__(*args,**kwargs)\n        hp=Path(self.output)/\'historico_nyu23.json\';self.history=c.read(hp) if hp.is_file() else []\n        self.add_callback(\'on_train_epoch_start\',self.begin);self.add_callback(\'on_model_save\',self.commit)\n    def get_validator(self):self.loss_names=(\'box_loss\',\'cls_loss\',\'dfl_loss\');return Validator(self)\n    def begin(self,trainer):self.epoch_started=time.monotonic()\n    def validate(self):\n        metrics=self.validator(self);fitness=metrics.pop(\'fitness\')\n        if not math.isfinite(fitness):raise FloatingPointError(\'Fitness não finito.\')\n        if self.best_fitness is None or fitness>self.best_fitness:self.best_fitness=fitness\n        return metrics,fitness\n    def save_model(self):\n        if any(not torch.isfinite(v).all() for v in self.ema.ema.state_dict().values() if v.is_floating_point()):raise FloatingPointError(\'EMA não finita; retomada anterior preservada.\')\n        return super().save_model()\n    def final_eval(self):return None  # Não retirar optimizer do checkpoint retomável.\n    def commit(self,trainer):\n        output=Path(self.output);epoch=self.epoch+1\n        previous=[r for r in self.history if r[\'epoch\']<epoch];old=max(previous,key=lambda r:(r[\'fitness\'],-r[\'epoch\'])) if previous else None\n        best=c.read(output/\'melhor_nyu23.json\') if (output/\'melhor_nyu23.json\').is_file() else None\n        if old is None or self.fitness>old[\'fitness\']:\n            target=output/\'pesos\'/f\'epoca_{epoch:03d}.pt\';export(self.last,target)\n            best={\'epoch\':epoch,\'fitness\':float(self.fitness),\'threshold\':self.pending[\'threshold\'],\'weight\':target.relative_to(output).as_posix(),\'weight_sha256\':c.sha(target),\'context\':self.context}\n            c.save(output/\'melhor_nyu23.json\',best)\n        row={\'epoch\':epoch,\'fitness\':float(self.fitness),\'selected\':{k:v for k,v in self.pending.items() if k!=\'image_rows\'},\'seconds\':time.monotonic()-self.epoch_started}\n        self.history=previous+[row];c.save(output/\'historico_nyu23.json\',self.history)\n        best_epoch=max(self.history,key=lambda r:(r[\'fitness\'],-r[\'epoch\']))[\'epoch\']\n        if epoch-best_epoch>=self.spec[\'patience\']:self.stop=True;self.reason=\'early_stopping\'\n        if epoch>=self.spec[\'epochs\']:self.reason=\'epochs_completed\'\n        elif self.reason is None and time.monotonic()-self.session_started>=self.spec[\'session_hours\']*3600:self.stop=True;self.reason=\'session_time_pause\'\n        slot=epoch%2;folder=output/\'retomada\';folder.mkdir(exist_ok=True);target=folder/f\'slot_{slot}.pt\';tmp=target.with_suffix(\'.tmp\');shutil.copy2(self.last,tmp);tmp.replace(target)\n        hs=folder/f\'historico_slot_{slot}.json\';c.save(hs,self.history)\n        c.save(folder/f\'slot_{slot}.json\',{\'epoch\':epoch,\'checkpoint\':target.name,\'sha256\':c.sha(target),\'context\':self.context,\n            \'history\':hs.name,\'history_sha256\':c.sha(hs),\'best\':best,\'reason\':self.reason})\n        det=self.pending[\'detector\'];print(f\'Época {epoch}/{self.spec["epochs"]}: P={det["precision"]}, R={det["recall"]}, confiança={self.pending["threshold"]}, meta={self.pending["target_met"]}\',flush=True)\n        print(f\'Época com validação: {row["seconds"]/60:.1f} min. Projeção máxima restante: {(self.spec["epochs"]-epoch)*row["seconds"]/3600:.1f} h.\',flush=True)\n        if self.backup:self.backup()\n        self.pending=None\n\ndef local_resume(output,context):\n    output=Path(output);options=[]\n    for path in (output/\'retomada\').glob(\'slot_*.json\'):\n        try:\n            m=c.read(path);weight=path.with_name(m[\'checkpoint\']);hs=path.with_name(m[\'history\']);best=output/m[\'best\'][\'weight\']\n            if m[\'context\']!=context:continue\n            if all(p.is_file() and c.sha(p)==s for p,s in [(weight,m[\'sha256\']),(hs,m[\'history_sha256\']),(best,m[\'best\'][\'weight_sha256\'])]):options.append((m,weight))\n        except (ValueError,KeyError,OSError):continue\n    return max(options,key=lambda x:x[0][\'epoch\']) if options else None\n\ndef sync(storage,output,remote):\n    output=Path(output)\n    try:\n        for p in sorted(output.rglob(\'*\')):\n            if not p.is_file() or p.suffix in {\'.tmp\',\'.part\',\'.zip\'} or \'__pycache__\' in p.parts:continue\n            rel=p.relative_to(output).as_posix()\n            if rel.startswith(\'treino/weights/\'):continue\n            storage.put(p,remote+\'/\'+rel)\n        return True\n    except Exception as exc:print(\'Backup incompleto; saída local preservada:\',type(exc).__name__,flush=True);return False\n\ndef restore(storage,output,remote,context):\n    output=Path(output);source=storage.path(remote)\n    if not source.is_dir():return None\n    for p in sorted(source.rglob(\'*\')):\n        if not p.is_file() or p.suffix in {\'.pt\',\'.zip\',\'.tmp\',\'.part\'} or p.name.endswith(\'.sha256.json\'):continue\n        rel=p.relative_to(source).as_posix()\n        if rel.startswith(\'treino/weights/\'):continue\n        storage.pull(remote+\'/\'+rel,output/rel)\n    options=[]\n    for p in (output/\'retomada\').glob(\'slot_*.json\'):\n        try:\n            m=c.read(p)\n            if m[\'context\']==context:options.append(m)\n        except (ValueError,KeyError):continue\n    for m in sorted(options,key=lambda x:x[\'epoch\'],reverse=True):\n        try:\n            storage.pull(remote+\'/retomada/\'+m[\'checkpoint\'],output/\'retomada\'/m[\'checkpoint\'],m[\'sha256\'])\n            storage.pull(remote+\'/retomada/\'+m[\'history\'],output/\'retomada\'/m[\'history\'],m[\'history_sha256\'])\n            storage.pull(remote+\'/\'+m[\'best\'][\'weight\'],output/m[\'best\'][\'weight\'],m[\'best\'][\'weight_sha256\'])\n        except Exception as exc:print(\'Slot incompleto; tentando o anterior:\',type(exc).__name__,flush=True);continue\n        c.save(output/\'historico_nyu23.json\',c.read(output/\'retomada\'/m[\'history\']));c.save(output/\'melhor_nyu23.json\',m[\'best\']);return m\n    if options:raise RuntimeError(\'Nenhum checkpoint remoto completo passou nas verificações. Os arquivos foram preservados.\')\n    return None\n\ndef train(initial,yaml_file,train_rows,val_rows,output,context,device,backup=None,spec=c.TRAIN_SPEC):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    if any(r[\'role\']!=\'train\' for r in train_rows) or any(r[\'role\']!=\'val\' for r in val_rows):raise ValueError(\'Papel proibido no treinamento.\')\n    frozen=output/\'decisao_antes_teste_nyu23.json\'\n    if frozen.is_file():\n        m=c.read(frozen)\n        if m[\'context\']!=context:raise ValueError(\'Decisão de outro contexto.\')\n        return {\'completed\':True,\'epochs\':m[\'epochs\'],\'status\':\'completed_frozen_reused\'}\n    prior=local_resume(output,context)\n    if prior:\n        c.save(output/\'historico_nyu23.json\',c.read(output/\'retomada\'/prior[0][\'history\']));c.save(output/\'melhor_nyu23.json\',prior[0][\'best\'])\n        if prior[0][\'epoch\']>=spec[\'epochs\'] or prior[0][\'reason\']==\'early_stopping\':return {\'completed\':True,\'epochs\':prior[0][\'epoch\'],\'status\':\'completed_reused\'}\n    elif list((output/\'retomada\').glob(\'slot_*.json\')):raise RuntimeError(\'Metadados existem, mas não há checkpoint completo validado.\')\n    class Configured(Trainer):pass\n    Configured.spec=spec;Configured.rows=val_rows;Configured.output=output;Configured.context=context;Configured.backup=staticmethod(backup) if backup else None\n    keys=[\'epochs\',\'patience\',\'imgsz\',\'batch\',\'workers\',\'optimizer\',\'lr0\',\'lrf\',\'warmup_epochs\',\'weight_decay\',\'cos_lr\',\'nbs\',\'amp\',\n          \'mosaic\',\'close_mosaic\',\'mixup\',\'cutmix\',\'copy_paste\',\'degrees\',\'translate\',\'scale\',\'fliplr\',\'flipud\',\'hsv_h\',\'hsv_s\',\'hsv_v\',\'seed\']\n    kwargs={k:spec[k] for k in keys};kwargs.update(data=str(yaml_file),device=device,project=str(output),name=\'treino\',save_dir=str(output/\'treino\'),\n        exist_ok=True,cache=False,compile=False,channels_last=False,deterministic=True,plots=False,save=True,val=True,verbose=False)\n    model=YOLO(str(prior[1] if prior else initial))\n    if not prior:\n        # O remapeamento nativo de linhas da cabeça usa nomes. Estes sinônimos\n        # preservam o pré-treino de sofá/mesa; rótulos NYU não são alterados.\n        aliases=spec[\'pretrained_name_aliases\']\n        model.model.names={i:aliases.get(name,name) for i,name in model.model.names.items()}\n    if prior:kwargs[\'resume\']=str(prior[1])\n    try:\n        model.train(trainer=Configured,**kwargs);t=model.trainer;paused=t.reason==\'session_time_pause\'\n        return {\'completed\':not paused,\'epochs\':t.epoch+1,\'status\':\'paused_resumable\' if paused else \'completed\',\'reason\':t.reason,\'resumed_from\':prior[0][\'epoch\'] if prior else None}\n    finally:\n        del model;gc.collect()\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n', 'nyu23_workflow.py': '"""Orquestra a preparação 22 e a sessão 23; não acessa pastas do experimento 21."""\nfrom __future__ import annotations\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport platform,shutil,time\nimport nyu22_common as c\nimport nyu22_data as data\n\ndef preparation(storage,local,output,bundle,spec=c.DATA_SPEC,sources=c.SOURCES):\n    local=Path(local);output=Path(output);local.mkdir(parents=True,exist_ok=True);output.mkdir(parents=True,exist_ok=True)\n    plan={\'spec\':spec,\'sources\':{k:sources[k] for k in [\'mat\',\'split\']},\'code_bundle\':bundle}\n    plan_key=c.digest(plan);remote=\'preparado/\'+plan_key[:24];ready=storage.path(remote+\'/pronto22.json\')\n    if ready.is_file():\n        info=c.read(ready)\n        if info[\'plan_key\']!=plan_key:raise ValueError(\'Preparação remota pertence a outro plano.\')\n        for key in [\'development\',\'reserve\']:\n            if not storage.path(remote+\'/\'+info[key][\'name\']).is_file():raise ValueError(\'A preparação anterior ficou sem um dos pacotes de dados.\')\n        for name in [\'preparacao_nyu22.json\',\'classes_desenvolvimento.csv\',\'exemplos_desenvolvimento.jpg\',\'pacote_preparacao_nyu22.zip\']:\n            if storage.path(remote+\'/\'+name).is_file():storage.pull(remote+\'/\'+name,output/name)\n        storage.put(ready,\'atual22.json\');return info\n    c.save(output/\'plano_preparacao_nyu22.json\',plan)\n    print(\'Baixando/conferindo o arquivo oficial anotado (~2,8 GiB). Não é o conjunto RAW de 428 GB.\',flush=True)\n    mat=storage.source(\'mat\',local/\'fontes\',sources);split=storage.source(\'split\',local/\'fontes\',sources)\n    report,dev,reserve=data.prepare(mat,split,local/\'base\',output,spec)\n    info={\'version\':spec[\'version\'],\'plan_key\':plan_key,\'remote\':remote,\'classes\':c.NAMES,\n        \'development\':{\'name\':dev.name,\'sha256\':c.sha(dev),\'bytes\':dev.stat().st_size},\n        \'reserve\':{\'name\':reserve.name,\'sha256\':c.sha(reserve),\'bytes\':reserve.stat().st_size},\n        \'sources\':sources,\'report\':report,\'test_used_for_selection\':False}\n    info[\'dataset_id\']=c.digest({\'plan_key\':plan_key,\'development\':info[\'development\'],\'reserve\':info[\'reserve\']})\n    c.save(output/\'pronto22.json\',info)\n    # A preparação só é publicada como atual depois dos dois pacotes completos.\n    storage.put(dev,remote+\'/\'+dev.name);storage.put(reserve,remote+\'/\'+reserve.name)\n    for name in [\'preparacao_nyu22.json\',\'classes_desenvolvimento.csv\',\'exemplos_desenvolvimento.jpg\',\'plano_preparacao_nyu22.json\']:\n        storage.put(output/name,remote+\'/\'+name)\n    archive=c.package(output,\'pacote_preparacao_nyu22.zip\');storage.put(archive,remote+\'/\'+archive.name)\n    storage.put(output/\'pronto22.json\',remote+\'/pronto22.json\');storage.put(output/\'pronto22.json\',\'atual22.json\')\n    return info\n\ndef session(storage,local,bundle,spec=c.TRAIN_SPEC,device=0):\n    import torch,ultralytics,numpy,PIL\n    import nyu23_training as tr\n    if ultralytics.__version__!=\'8.4.150\':raise RuntimeError(\'A versão importada difere da fixada. Reinicie a sessão e use Run All.\')\n    local=Path(local);info=c.read(storage.path(\'atual22.json\'))\n    if info[\'classes\']!=c.NAMES:raise ValueError(\'Execute a preparação 22 desta linha antes do treinamento 23.\')\n    base=local/\'base\'/info[\'dataset_id\'][:24];base.mkdir(parents=True,exist_ok=True)\n    archive=storage.pull(info[\'remote\']+\'/\'+info[\'development\'][\'name\'],base/\'desenvolvimento.zip\',info[\'development\'][\'sha256\'])\n    train,val,yaml_file,manifest=data.load_development(archive,base)\n    identity={\'version\':spec[\'version\'],\'code_bundle\':bundle,\'dataset_id\':info[\'dataset_id\'],\'spec\':spec,\'ultralytics\':\'8.4.150\',\'pretrained_sha256\':c.SOURCES[\'weight\'][\'sha256\']}\n    context=c.digest(identity);output=local/\'resultados\'/context[:24];output.mkdir(parents=True,exist_ok=True);remote=\'treinos/\'+context[:24]\n    if tr.local_resume(output,context) is None:tr.restore(storage,output,remote,context)\n    c.save(output/\'configuracao_nyu23.json\',{\'context\':context,**identity});c.save(output/\'base_nyu22.json\',info)\n    shutil.copy2(base/\'manifesto_desenvolvimento.json\',output/\'manifesto_desenvolvimento_referencia.json\')\n    stamp=datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S\')\n    environment={\'python\':platform.python_version(),\'torch\':torch.__version__,\'numpy\':numpy.__version__,\'pillow\':PIL.__version__,\n        \'ultralytics\':ultralytics.__version__,\'device\':str(device),\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n        \'cuda\':torch.version.cuda,\'amp\':False,\'quantize\':32}\n    c.save(output/\'sessoes\'/(stamp+\'.json\'),environment)\n    initial=storage.source(\'weight\',local/\'fontes\')\n    return {\'context\':context,\'output\':output,\'remote\':remote,\'train\':train,\'val\':val,\'yaml\':yaml_file,\'manifest\':manifest,\'info\':info,\'base\':base,\'initial\':initial,\'environment\':environment}\n\ndef evaluate_session(storage,job,run,device,spec=c.TRAIN_SPEC,backup=None):\n    from ultralytics import YOLO\n    import nyu23_eval as ev\n    output=job[\'output\'];best=c.read(output/\'melhor_nyu23.json\');weight=output/best[\'weight\']\n    if best[\'context\']!=job[\'context\'] or c.sha(weight)!=best[\'weight_sha256\']:raise ValueError(\'Melhor peso alterado ou de outra execução.\')\n    shutil.copy2(weight,output/\'melhor_modelo.pt\');model=YOLO(str(weight))\n    decision_path=output/\'decisao_antes_teste_nyu23.json\';frozen=c.read(decision_path) if decision_path.is_file() else None\n    if frozen and (frozen[\'context\']!=job[\'context\'] or frozen[\'weight_sha256\']!=best[\'weight_sha256\']):raise ValueError(\'A decisão já congelada aponta para outro modelo.\')\n    selected,profiles=ev.evaluate_development(model,job[\'val\'],device,spec,output/\'validacao\',best[\'weight_sha256\'],frozen[\'threshold\'] if frozen else None)\n    test=None\n    if run[\'completed\']:\n        if frozen is None:\n            frozen={\'context\':job[\'context\'],\'weight_sha256\':best[\'weight_sha256\'],\'threshold\':selected[\'threshold\'],\'epochs\':run[\'epochs\'],\n                \'classes\':c.NAMES,\'dataset_id\':job[\'info\'][\'dataset_id\'],\'inference\':{k:spec[k] for k in [\'imgsz\',\'inference_conf\',\'nms_iou\',\'match_iou\',\'max_det\']},\n                \'decision_uses_test\':False,\'selection\':\'Somente VAL; peso e limiar fixados antes de consultar previsões de TEST.\'}\n            c.save(decision_path,frozen)\n        # Falha neste envio interrompe antes da primeira previsão na reserva.\n        storage.put(decision_path,job[\'remote\']+\'/\'+decision_path.name)\n        test_dir=output/\'teste_final\';cached=test_dir/\'metricas.json\';frozen_key=c.digest(frozen)\n        if cached.is_file():\n            test=c.read(cached)\n            if test[\'frozen_key\']!=frozen_key or not test[\'complete\']:raise ValueError(\'Resultado de TEST não corresponde à decisão congelada.\')\n            print(\'Teste final já concluído: reutilizando o resultado congelado.\',flush=True)\n        else:\n            info=job[\'info\'];archive=storage.pull(info[\'remote\']+\'/\'+info[\'reserve\'][\'name\'],job[\'base\']/\'reserva.zip\',info[\'reserve\'][\'sha256\'])\n            rows=data.load_reserve(archive,job[\'base\'],job[\'manifest\'])\n            c.save(output/\'abertura_teste_nyu23.json\',{\'frozen_key\':frozen_key,\'images\':len(rows),\'weights_or_thresholds_changed\':False})\n            predictions=ev.obtain_cache(model,rows,device,spec,test_dir/\'cache\',best[\'weight_sha256\'],\'test\',checkpoint=backup)\n            test=ev.profile(rows,predictions,frozen[\'threshold\'],spec);test[\'confidence_intervals\']=ev.scene_bootstrap(test[\'image_rows\'],spec)\n            test_yaml=data.write_yaml(job[\'train\'],rows,job[\'base\'],\'test_data.yaml\')\n            native=model.val(data=str(test_yaml),imgsz=spec[\'imgsz\'],batch=spec[\'predict_batch\'],device=device,\n                conf=.001,iou=spec[\'nms_iou\'],max_det=spec[\'max_det\'],quantize=32,workers=spec[\'workers\'],\n                plots=False,save_json=False,verbose=False,project=str(output),name=\'ap_teste\',exist_ok=True)\n            test.update(native_ap={\'map50\':float(native.box.map50),\'map50_95\':float(native.box.map)},frozen_key=frozen_key,complete=True)\n            c.save(cached,test);c.csv_save(test_dir/\'classes.csv\',test[\'classes\']);c.csv_save(test_dir/\'imagens.csv\',test[\'image_rows\'])\n    del model\n    return best,selected,test\n\ndef failure(output,error,stage,storage=None,remote=None):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    c.save(output/\'resumo_nyu23.json\',{\'completed\':False,\'status\':\'interrompido\',\'stage\':stage,\'error\':{\'type\':type(error).__name__,\'message\':str(error)[:1600]},\'manual_form_required\':False})\n    archive=c.package(output,\'pacote_diagnostico_nyu.zip\')\n    if storage and remote:\n        try:storage.put(archive,remote+\'/\'+archive.name)\n        except Exception:pass\n    return archive\n'}
HASHES={'nyu22_common.py': '0e75c263b8396c01fe5f9e011de032e85989f5198290ceadf71848e97cb08cb2', 'nyu22_data.py': '423dfb6b66ec65f5338be2ba8cb6bdabb95091fe8239f698931d130e3eb1b480', 'nyu23_eval.py': 'ab7b7b96749f01cb4dff2f6a2438f15845c4ffca3433057aa13722b873166662', 'nyu23_report.py': 'b799315a7aff549461668eed6dcb2f0d67a2b327e614c575c90d62678ade6482', 'nyu23_training.py': 'aa608a0f952dd128b183186e0c4a0718c1e9689b97b621038e5472f0b4748116', 'nyu23_workflow.py': '2d4a8bf542f99a49a28d4c8de40d0a18ad37ac08669ec530a51937ea61c98f29'}

LOCAL=Path('/content/tcc_nyu22')
BOOT=LOCAL/'saida_inicial';CODE=LOCAL/'codigo'
LOCAL.mkdir(parents=True,exist_ok=True);BOOT.mkdir(exist_ok=True);CODE.mkdir(exist_ok=True)
for name,source in MODULOS.items():
    if hashlib.sha256(source.encode()).hexdigest()!=HASHES[name]:raise RuntimeError('Código embutido alterado: '+name)
    compile(source,name,'exec')
    for root in [CODE,BOOT/'codigo']:
        root.mkdir(parents=True,exist_ok=True);(root/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(CODE))
import nyu22_common as c
import nyu23_workflow as workflow

from google.colab import drive
drive.mount('/content/drive')
STORAGE=c.Storage(Path('/content/drive/MyDrive')/c.BRANCH)
OUTPUT=BOOT;REMOTE='diagnosticos/inicial_22'
def falha(exc,stage):
    archive=workflow.failure(OUTPUT,exc,stage,STORAGE,REMOTE)
    print('Execução interrompida. Pacote de diagnóstico:',archive)
    display(FileLink(str(archive)))
def baixar(archive):
    try:
        from google.colab import files
        files.download(str(archive))
    except Exception:
        print('O download automático não abriu. O pacote continua na saída local e no Drive:',archive)


## 2. Baixar, converter e separar os dados

As seis classes são resolvidas pelos nomes do arquivo oficial. Cada par
de classe original e número de instância produz uma caixa, incluindo a
instância zero quando há rótulo válido, conforme o toolbox NYU.

Imagens da mesma cena, inclusive sequências com sufixos, permanecem juntas.
Semelhanças pHash entre cenas também unem grupos dentro do TRAIN oficial.
A divisão tenta preservar suporte de classes antes de qualquer treino,
sem usar métricas de modelos nem os rótulos da reserva para escolher grupos.

A reserva é convertida para outro ZIP; essa conversão não executa modelo
nem escolhe parâmetros. O notebook 23 só abre suas previsões depois de
congelar o melhor peso e a confiança.


In [ ]:
try:
    INFO=workflow.preparation(STORAGE,LOCAL,OUTPUT,BUNDLE)
    PREP=INFO['report'];REMOTE=INFO['remote']
    print('PREPARAÇÃO 22 CONCLUÍDA')
    print('TRAIN:',PREP['train_images'],'| VAL:',PREP['val_images'],'| TEST reservado:',PREP['test_images_reserved'])
    print('Classes:',c.PT_NAMES)
    print('Sobreposição de cenas TRAIN/VAL:',PREP['scene_overlap_train_val'])
    print('Sobreposição de cenas desenvolvimento/TEST:',PREP['scene_overlap_development_test'])
    display(Image(filename=str(OUTPUT/'exemplos_desenvolvimento.jpg')))
except Exception as exc:
    falha(exc,'22-preparacao');raise


## 3. Salvar o pacote e seguir para o notebook 23

A base preparada e a reserva estão na pasta própria deste ramo no Drive.
O pacote abaixo é um relatório leve com fontes, contagens e exemplos de
desenvolvimento; não inclui o MAT de 2,8 GiB nem todas as imagens.

**Agora abra o notebook 23 em uma sessão Colab com GPU e execute tudo.**
A autorização do Drive pode ser solicitada novamente. O notebook 21 pode
continuar no Kaggle em paralelo.


In [ ]:
try:
    PACOTE=OUTPUT/'pacote_preparacao_nyu22.zip'
    if not PACOTE.is_file():PACOTE=c.package(OUTPUT,PACOTE.name)
    print('Pacote de preparação:',PACOTE)
    print('Próximo: 23_nyu_compacto_treinamento_colab.ipynb, com GPU.')
    baixar(PACOTE)
except Exception as exc:
    falha(exc,'22-pacote');raise
